# Spoiler Shield: sentence-level spoiler models

**Question:** can we flag the individual spoiler *sentences* in a movie review, and does giving the
model the movie's plot synopsis help compared to reading the review sentence alone?

**Data:** `data/spoiler_detection_dataset_1.3.1.json`: 80 movies, 3,200 IMDb reviews, 60,189 sentences
with sentence-level labels, fixed train/test split by movie (59 / 21 movies). Plot synopses come from
`data/plot_synopses.json`.

**Model ladder** (all evaluated on the same 21 unseen test movies):

| ID | Model | Sees the synopsis? | Where it runs |
|---|---|---|---|
| A | TF-IDF (word + char n-grams) + logistic regression | no | CPU, ~1 min |
| B0 | Similarity threshold: max TF-IDF cosine to a *late* synopsis sentence, no training | yes | CPU |
| B | Logistic regression on synopsis-matching features only | yes | CPU |
| C | A + B features together | yes | CPU |
| D | DistilBERT fine-tuned on the sentence only | no | GPU (Colab), ~10 min |
| E | DistilBERT on (sentence, best-matching synopsis sentences) pairs | yes | GPU (Colab), ~25 min |

The ablations A vs C and D vs E answer the key question: does the plot really help?

**How to run:** *Runtime → Run all*. In Google Colab, switch to a GPU first (*Runtime → Change runtime
type → T4 GPU*); the notebook clones the repository by itself. Without a GPU the DistilBERT
section is skipped and only A–C run. All results are written to `results/`.

## 0. Setup

In [ ]:
import os, sys, re, json, random, time, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/paulcloud17/sentence_level_spoiler_shield.git"


def find_repo_root():
    for cand in [Path.cwd(), Path.cwd().parent, Path("/content/sentence_level_spoiler_shield")]:
        if (cand / "data" / "spoiler_detection_dataset_1.3.1.json").exists():
            return cand
    return None


if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"], check=False)

ROOT = find_repo_root()
if ROOT is None:  # e.g. a fresh Colab runtime
    subprocess.run(["git", "clone", "-q", REPO_URL, "/content/sentence_level_spoiler_shield"], check=True)
    ROOT = find_repo_root()
DATA = ROOT / "data"
RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)
print("repo root:", ROOT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_recall_curve,
                             precision_score, recall_score, f1_score)

SEED = 42
random.seed(SEED); np.random.seed(SEED)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 200)

## 1. Data
One row per sentence. We keep the movie, the review and the sentence's position in the review, so that
all splits are by movie and all masking metrics can be computed per review.

In [ ]:
raw = json.load(open(DATA / "spoiler_detection_dataset_1.3.1.json"))
synopses = json.load(open(DATA / "plot_synopses.json"))

rows = []
for r_idx, r in enumerate(raw["data"]):
    n = len(r["sentences"])
    for s_idx, s in enumerate(r["sentences"]):
        rows.append(dict(
            review_id=r_idx, movie_id=r["movie_id"], split=r["split"],
            is_spoiler_review=r["is_spoiler_review"], s_idx=s_idx, s_pos=s_idx / max(n - 1, 1),
            text=s["text"], y=int(s["is_spoiler"]), unsure=bool(s.get("unsure", False)),
            annotation_source=r["annotation_source"],
        ))
df = pd.DataFrame(rows)
df["n_words"] = df.text.str.split().str.len()
print(f"{len(df):,} sentences, {df.review_id.nunique():,} reviews, {df.movie_id.nunique()} movies")
df.groupby("split").agg(movies=("movie_id", "nunique"), reviews=("review_id", "nunique"),
                        sentences=("y", "size"), spoilers=("y", "sum"), spoiler_rate=("y", "mean"),
                        unsure=("unsure", "sum"))

**Who labelled what.** Train labels were produced by an LLM (Claude Sonnet, given the synopsis) and
1,553 of them are flagged `unsure`; test labels were reviewed by the team. So the test set is our
gold standard, and train labels contain some noise.

**The reviewer's own IMDb spoiler flag is a weak label.** The table below shows how it relates to
whether the review contains at least one spoiler sentence: many reviews flagged as spoilers contain
none, and some unflagged ones do. This is why we annotate sentences instead of trusting the flag.

In [ ]:
rev = df.groupby("review_id").agg(split=("split", "first"), flag=("is_spoiler_review", "first"),
                                  has_spoiler_sentence=("y", "max"), n_sent=("y", "size"))
pd.crosstab([rev.split, rev.flag], rev.has_spoiler_sentence, margins=True)

In [ ]:
print(df.annotation_source.value_counts(), "\n")
print("sentence length (words):", df.n_words.describe().round(1).to_dict())
print("\nspoiler rate by position in the review (5 bins):")
print(df.groupby(pd.cut(df.s_pos, 5, include_lowest=True), observed=True).y.mean().round(3))

**Validation set.** The test set is only touched once, for the final numbers. All tuning
(regularisation, decision threshold, early stopping) uses 12 *train* movies held out as validation.

In [ ]:
train_all = df[df.split == "train"].copy()
test = df[df.split == "test"].reset_index(drop=True)
gss = GroupShuffleSplit(n_splits=1, test_size=12 / 59, random_state=SEED)
tr_idx, va_idx = next(gss.split(train_all, groups=train_all.movie_id))
train = train_all.iloc[tr_idx].reset_index(drop=True)
val = train_all.iloc[va_idx].reset_index(drop=True)
for name, part in [("train", train), ("val", val), ("test", test)]:
    print(f"{name:5s} movies={part.movie_id.nunique():2d} sentences={len(part):6,d} spoiler rate={part.y.mean():.3f}")
assert not set(train.movie_id) & set(test.movie_id) and not set(val.movie_id) & set(test.movie_id)

## 2. Plot synopses
Each synopsis is split into sentences and every synopsis sentence gets its relative position in the plot
(0 = opening, 1 = ending). Position matters: matching a review sentence to the *end* of the plot is a much
stronger spoiler signal than matching it to the premise.

One test movie (tt0369610, *Jurassic World*) has an empty synopsis in the source data; we fall back to
its short plot summary, so synopsis-based models are handicapped on that movie.

In [ ]:
SENT_SPLIT = re.compile(r"(?<=[.!?])\s+(?=[\"'(\[]?[A-Z0-9])")


def split_sentences(text):
    return [s.strip() for s in SENT_SPLIT.split(re.sub(r"\s+", " ", text or "")) if len(s.strip()) > 3]


syn_sents = {}  # movie_id -> list of (sentence, relative position)
for m in df.movie_id.unique():
    info = synopses.get(m, {})
    text = info.get("plot_synopsis") or ""
    if len(text.split()) < 50:
        text = re.sub(r"\s*Written by.*$", "", info.get("plot_summary") or "", flags=re.S)
        print(f"{m}: synopsis missing, using plot_summary ({len(text.split())} words)")
    ss = split_sentences(text)
    syn_sents[m] = [(s, i / max(len(ss) - 1, 1)) for i, s in enumerate(ss)]
print("synopsis sentences per movie:", pd.Series({m: len(v) for m, v in syn_sents.items()}).describe().round(0).to_dict())

## 3. Evaluation
We use metrics of different character, because one number hides the trade-offs that matter for masking:

* **PR-AUC** (average precision), threshold-free and suitable for the 6–8% positive rate. A random
  model scores ≈ the positive rate (0.063 on test). We add a 95% bootstrap interval that **resamples
  movies**, since sentences within a movie are not independent and we only have 21 test movies.
* **ROC-AUC**, as a ranking measure that is insensitive to the class ratio.
* At the **operating threshold** (chosen on validation to maximise F1): precision, recall, F1 on the
  spoiler class.
* **Recall @ precision ≥ 0.5**: how many spoilers we catch if at most half of the masked sentences may be
  false alarms.
* **Masking metrics** (what a reader experiences): share of words hidden, share of spoiler sentences
  left visible, share of spoiler-containing reviews fully cleaned, share of spoiler-free reviews left untouched.

In [ ]:
def best_f1_threshold(y, s):
    p, r, t = precision_recall_curve(y, s)
    f = 2 * p * r / np.clip(p + r, 1e-9, None)
    return float(t[np.nanargmax(f[:-1])])


def recall_at_precision(y, s, min_p=0.5):
    p, r, _ = precision_recall_curve(y, s)
    ok = p >= min_p
    return float(r[ok].max()) if ok.any() else 0.0


def bootstrap_pr_auc(frame, score_col, n=1000, seed=SEED):
    rng = np.random.default_rng(seed)
    movies = frame.movie_id.unique()
    groups = {m: g for m, g in frame.groupby("movie_id")}
    vals = []
    for _ in range(n):
        sample = pd.concat([groups[m] for m in rng.choice(movies, len(movies))])
        if sample.y.sum() > 0:
            vals.append(average_precision_score(sample.y, sample[score_col]))
    return np.percentile(vals, [2.5, 97.5])


def evaluate(name, frame, score_col, thr):
    y, s = frame.y.values, frame[score_col].values
    pred = s >= thr
    lo, hi = bootstrap_pr_auc(frame, score_col)
    g = frame.assign(pred=pred, hit=pred & (y == 1)).groupby("review_id")
    per_rev = g.agg(has_sp=("y", "max"), n_sp=("y", "sum"), n_hit=("hit", "sum"), any_mask=("pred", "max"))
    per_rev["missed"] = per_rev.n_sp - per_rev.n_hit
    return dict(
        model=name,
        pr_auc=average_precision_score(y, s), pr_auc_ci=f"[{lo:.3f}, {hi:.3f}]",
        roc_auc=roc_auc_score(y, s), threshold=thr,
        precision=precision_score(y, pred, zero_division=0), recall=recall_score(y, pred),
        f1=f1_score(y, pred), recall_at_p50=recall_at_precision(y, s, 0.5),
        words_masked=(pred * frame.n_words).sum() / frame.n_words.sum(),
        spoilers_missed=1 - recall_score(y, pred),
        spoiler_reviews_fully_clean=(per_rev[per_rev.has_sp == 1].missed == 0).mean(),
        clean_reviews_untouched=(per_rev[per_rev.has_sp == 0].any_mask == 0).mean(),
    )


results = []          # one dict per model
SCORE_COLS = {}       # model name -> column with its test score
print("positive rate (= PR-AUC of a random model): val %.3f, test %.3f" % (val.y.mean(), test.y.mean()))

## 4. Model A: TF-IDF + logistic regression (review sentence only)
Word 1–2-grams catch phrases like *"in the end"*, *"turns out"*, *"dies"*; character 3–5-grams are robust
to spelling and names. `class_weight="balanced"` compensates for the 8% positive rate. `C` is tuned on validation.

In [ ]:
word_vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=100_000, sublinear_tf=True, strip_accents="unicode")
char_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_features=150_000, sublinear_tf=True)
Xw_tr = word_vec.fit_transform(train.text); Xc_tr = char_vec.fit_transform(train.text)
def tfidf(frame):
    return sparse.hstack([word_vec.transform(frame.text), char_vec.transform(frame.text)]).tocsr()
XA_tr, XA_va, XA_te = sparse.hstack([Xw_tr, Xc_tr]).tocsr(), tfidf(val), tfidf(test)


def fit_lr(X_tr, X_va, Cs=(0.1, 0.3, 1, 3, 10)):
    best = None
    for C in Cs:
        m = LogisticRegression(C=C, class_weight="balanced", max_iter=3000)
        m.fit(X_tr, train.y)
        ap = average_precision_score(val.y, m.predict_proba(X_va)[:, 1])
        print(f"  C={C:<4} val PR-AUC={ap:.3f}")
        if best is None or ap > best[0]:
            best = (ap, C, m)
    print(f"  -> best C={best[1]}")
    return best[2]


lrA = fit_lr(XA_tr, XA_va)
val["score_A"] = lrA.predict_proba(XA_va)[:, 1]
test["score_A"] = lrA.predict_proba(XA_te)[:, 1]
thrA = best_f1_threshold(val.y, val.score_A)
results.append(evaluate("A  TF-IDF + LR (review only)", test, "score_A", thrA)); SCORE_COLS["A"] = "score_A"
results[-1]

What did model A learn? The strongest n-grams in each direction:

In [ ]:
names = np.r_[word_vec.get_feature_names_out(), char_vec.get_feature_names_out()]
coef = lrA.coef_[0]
nw = len(word_vec.get_feature_names_out())
top_w = np.argsort(coef[:nw])
print("SPOILER words:    ", ", ".join(names[top_w[::-1][:30]]))
print("NON-SPOILER words:", ", ".join(names[top_w[:30]]))

## 5. Synopsis matching features (models B0, B, C)
For every review sentence we compare it to every sentence of *its own movie's* synopsis (TF-IDF cosine,
vectoriser fitted on train reviews + synopses) and compute 10 features:

* `max_sim`, `top3_sim`: how strongly the sentence matches any plot sentence;
* `late_max` / `early_max`: best match in the last 80% vs. the first 20% of the plot (setup = not a spoiler,
  as in the annotation guidelines);
* `best_pos`, `wpos`: plot position of the best match and similarity-weighted position;
* `n_match`: number of plot sentences with cosine > 0.15;
* `name_hits`, `late_names`: capitalised words shared with the synopsis, overall and in the late plot (character names);
* `log_len`: sentence length.

The index of the best-matching synopsis sentence is kept: it is the **explanation** shown to the user ("this sentence
was masked because it matches this part of the plot").

In [ ]:
syn_vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, stop_words="english", strip_accents="unicode")
syn_vec.fit(pd.concat([train.text, pd.Series([s for v in syn_sents.values() for s, _ in v])]))
syn_mats = {m: (syn_vec.transform([s for s, _ in v]), np.array([p for _, p in v]),
                [set(re.findall(r"\b[A-Z][a-z]{2,}\b", s)) for s, _ in v])
            for m, v in syn_sents.items()}
SYN_FEATS = ["max_sim", "top3_sim", "late_max", "early_max", "best_pos", "wpos", "n_match", "name_hits", "late_names", "log_len"]
COMMON_CAPS = set("The This That They There Then What When While After Before His Her It Its In On At And But So If As For "
                  "One All Some Most Even Also Just Only Not No Yes Well Overall However Although Though I We You He She Who "
                  "Why How Where Which Also Movie Film".split())


def synopsis_features(frame):
    feats = np.zeros((len(frame), len(SYN_FEATS))); best_idx = np.zeros(len(frame), dtype=int)
    for m, g in frame.groupby("movie_id"):
        S, pos, caps = syn_mats[m]
        sims = (syn_vec.transform(g.text) @ S.T).toarray()          # (n_review_sents, n_syn_sents)
        late, early = pos > 0.2, pos <= 0.2
        order = np.argsort(-sims, axis=1)
        rows_pos = frame.index.get_indexer(g.index)
        for k, (row_i, text) in enumerate(zip(rows_pos, g.text)):
            s = sims[k]; b = order[k, 0]
            words = set(re.findall(r"\b[A-Z][a-z]{2,}\b", text)) - COMMON_CAPS
            hits = [len(words & c) for c in caps]
            feats[row_i] = [s[b], s[order[k, :3]].mean(),
                            s[late].max() if late.any() else 0, s[early].max() if early.any() else 0,
                            pos[b] if s[b] > 0 else 0, (s * pos).sum() / s.sum() if s.sum() > 0 else 0,
                            (s > 0.15).sum(), len(words & set().union(*caps)) if caps else 0,
                            max([h for h, p in zip(hits, pos) if p > 0.2], default=0), np.log1p(len(text.split()))]
            best_idx[row_i] = b
    return pd.DataFrame(feats, columns=SYN_FEATS, index=frame.index), best_idx


t0 = time.time()
F_tr, _ = synopsis_features(train)
F_va, _ = synopsis_features(val)
F_te, best_te = synopsis_features(test)
test["best_syn_idx"] = best_te
test["best_syn_sentence"] = [syn_sents[m][i][0] if syn_sents[m] else "" for m, i in zip(test.movie_id, best_te)]
test["best_syn_pos"] = [syn_sents[m][i][1] if syn_sents[m] else 0 for m, i in zip(test.movie_id, best_te)]
print(f"features computed in {time.time() - t0:.0f}s")
F_tr.assign(y=train.y).groupby("y").mean().T.round(3)

**B0: similarity threshold, no learning.** The score is just `late_max`, the cosine to the best-matching
sentence in the post-setup part of the plot. This is the simplest possible "does it repeat the plot?" detector.

In [ ]:
val["score_B0"], test["score_B0"] = F_va.late_max.values, F_te.late_max.values
thrB0 = best_f1_threshold(val.y, val.score_B0)
results.append(evaluate("B0 Synopsis similarity threshold", test, "score_B0", thrB0)); SCORE_COLS["B0"] = "score_B0"

**B: logistic regression on the 10 synopsis features only.**

In [ ]:
scaler = StandardScaler().fit(F_tr)
FS_tr, FS_va, FS_te = (scaler.transform(F) for F in (F_tr, F_va, F_te))
lrB = fit_lr(FS_tr, FS_va, Cs=(0.01, 0.1, 1, 10))
val["score_B"], test["score_B"] = lrB.predict_proba(FS_va)[:, 1], lrB.predict_proba(FS_te)[:, 1]
thrB = best_f1_threshold(val.y, val.score_B)
results.append(evaluate("B  LR on synopsis features", test, "score_B", thrB)); SCORE_COLS["B"] = "score_B"
print(pd.Series(lrB.coef_[0], index=SYN_FEATS).sort_values().round(2))

**C: review text + synopsis features.** Same TF-IDF as A plus the scaled synopsis features. A vs C is the
classical-model answer to *"does the synopsis help?"*. The synopsis block is up-weighted so that 10 dense
features are not drowned out by 250k sparse ones; the weight is chosen on validation.

In [ ]:
best = None
for w in (0.5, 1, 2, 4):
    XC_tr = sparse.hstack([XA_tr, sparse.csr_matrix(FS_tr * w)]).tocsr()
    XC_va = sparse.hstack([XA_va, sparse.csr_matrix(FS_va * w)]).tocsr()
    print(f"synopsis weight {w}:")
    m = fit_lr(XC_tr, XC_va, Cs=(0.3, 1, 3))
    ap = average_precision_score(val.y, m.predict_proba(XC_va)[:, 1])
    if best is None or ap > best[0]:
        best = (ap, w, m)
_, wC, lrC = best
print("chosen synopsis weight:", wC)
XC_va = sparse.hstack([XA_va, sparse.csr_matrix(FS_va * wC)]).tocsr()
XC_te = sparse.hstack([XA_te, sparse.csr_matrix(FS_te * wC)]).tocsr()
val["score_C"], test["score_C"] = lrC.predict_proba(XC_va)[:, 1], lrC.predict_proba(XC_te)[:, 1]
thrC = best_f1_threshold(val.y, val.score_C)
results.append(evaluate("C  TF-IDF + synopsis features", test, "score_C", thrC)); SCORE_COLS["C"] = "score_C"
pd.DataFrame(results).set_index("model").round(3)

## 6. DistilBERT (models D and E), GPU only

* **D: review only:** input is the sentence alone (max 64 tokens).
* **E: review + synopsis:** input is a sentence pair: `[CLS] review sentence [SEP] synopsis context [SEP]`, where the
  context is the 3 synopsis sentences of the same movie that match best, each prefixed with its place in the
  plot (`(beginning)`, `(middle)`, `(end)`) so the model can learn that matching the ending matters most (max 256 tokens).
  Retrieval uses the sentence-embedding model `all-MiniLM-L6-v2` (paraphrase-aware); if it is not available,
  it falls back to the TF-IDF matching from section 5.

Both use the same training recipe: 3 epochs, AdamW (lr 3e-5, linear warm-up/decay), batch 32, class-weighted
cross-entropy (spoiler weight 3), best epoch kept by validation PR-AUC, threshold tuned on validation.
A plain PyTorch loop is used instead of `Trainer` so it does not break across `transformers` versions.

In [ ]:
try:
    import torch
    HAS_GPU = torch.cuda.is_available()
except ImportError:
    HAS_GPU = False
RUN_TRANSFORMERS = HAS_GPU          # set to True to force it on CPU (slow: hours)
SMOKE_TEST = False                  # True = tiny random model on a few hundred sentences, only to check the code runs
MODEL_NAME = "distilbert-base-uncased"
print("GPU available:", HAS_GPU, "| running DistilBERT:", RUN_TRANSFORMERS or SMOKE_TEST)

In [ ]:
if RUN_TRANSFORMERS or SMOKE_TEST:
    import torch
    from torch.utils.data import DataLoader
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(SEED)

    def plot_bucket(p):
        return "(beginning)" if p <= 0.2 else "(end)" if p >= 0.75 else "(middle)"

    def retrieve_context(frame, k=3):
        """For each review sentence: its k best-matching synopsis sentences, with plot-position tags."""
        try:
            if SMOKE_TEST:
                raise RuntimeError("smoke test: use TF-IDF retrieval")
            global _ENC, _SYN_EMB
            if "_ENC" not in globals():
                from sentence_transformers import SentenceTransformer
                _ENC = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=DEVICE)
                _SYN_EMB = {m: _ENC.encode([s for s, _ in v], normalize_embeddings=True, batch_size=128)
                            for m, v in syn_sents.items() if v}
            enc, syn_emb = _ENC, _SYN_EMB
            q = enc.encode(frame.text.tolist(), normalize_embeddings=True, batch_size=256, show_progress_bar=True)
            sims_for = lambda i, m: syn_emb[m] @ q[i]
            print("retrieval: all-MiniLM-L6-v2")
        except Exception as e:
            print("retrieval: TF-IDF fallback (", e, ")")
            Q = syn_vec.transform(frame.text)
            sims_for = lambda i, m: (syn_mats[m][0] @ Q[i].T).toarray().ravel()
        ctx = []
        for i, m in enumerate(frame.movie_id.values):
            v = syn_sents[m]
            if not v:
                ctx.append(""); continue
            top = np.argsort(-sims_for(i, m))[:k]
            ctx.append(" ".join(f"{plot_bucket(v[j][1])} {v[j][0]}" for j in sorted(top)))
        return ctx

    class SentDS(torch.utils.data.Dataset):
        def __init__(self, enc, y):
            self.enc, self.y = enc, y
        def __len__(self):
            return len(self.y)
        def __getitem__(self, i):
            item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
            item["labels"] = torch.tensor(self.y[i])
            return item

    def collate(batch, pad_id):
        L = max(len(b["input_ids"]) for b in batch)
        out = {}
        for k in batch[0]:
            if k == "labels":
                out[k] = torch.stack([b[k] for b in batch])
            else:
                fill = pad_id if k == "input_ids" else 0
                out[k] = torch.stack([torch.cat([b[k], torch.full((L - len(b[k]),), fill, dtype=b[k].dtype)]) for b in batch])
        return out

    def load_model_and_tokenizer():
        if SMOKE_TEST:
            from smoke_helpers import tiny_model_and_tokenizer  # local helper, not needed in Colab
            return tiny_model_and_tokenizer(train.text.tolist())
        tok = AutoTokenizer.from_pretrained(MODEL_NAME)
        model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
        return model, tok

    @torch.no_grad()
    def predict(model, loader):
        model.eval(); out = []
        for b in loader:
            b = {k: v.to(DEVICE) for k, v in b.items() if k != "labels"}
            with torch.autocast(DEVICE, enabled=DEVICE == "cuda"):
                logits = model(**b).logits
            out.append(torch.softmax(logits.float(), -1)[:, 1].cpu().numpy())
        return np.concatenate(out)

    def train_transformer(tag, tr, va, te, pair_cols=None, max_len=64, epochs=3, lr=3e-5, bs=32, pos_weight=3.0):
        model, tok = load_model_and_tokenizer()
        model.to(DEVICE)
        def encode(frame):
            if pair_cols:
                return tok(frame.text.tolist(), frame[pair_cols].tolist(), truncation="longest_first", max_length=max_len)
            return tok(frame.text.tolist(), truncation=True, max_length=max_len)
        enc = {n: encode(f) for n, f in [("tr", tr), ("va", va), ("te", te)]}
        mk = lambda n, f, shuffle: DataLoader(SentDS(dict(enc[n]), f.y.values), batch_size=bs if shuffle else bs * 4,
                                              shuffle=shuffle, collate_fn=lambda b: collate(b, tok.pad_token_id))
        dl_tr, dl_va, dl_te = mk("tr", tr, True), mk("va", va, False), mk("te", te, False)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
        steps = epochs * len(dl_tr)
        sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
        scaler_amp = torch.amp.GradScaler(enabled=DEVICE == "cuda")
        loss_fn = torch.nn.CrossEntropyLoss(weight=torch.tensor([1.0, pos_weight], device=DEVICE))
        best = (-1, None)
        history = []
        for ep in range(epochs):
            model.train(); t0 = time.time(); tot = 0
            for i, b in enumerate(dl_tr):
                b = {k: v.to(DEVICE) for k, v in b.items()}
                y = b.pop("labels")
                with torch.autocast(DEVICE, enabled=DEVICE == "cuda"):
                    loss = loss_fn(model(**b).logits.float(), y)
                opt.zero_grad(); scaler_amp.scale(loss).backward()
                scaler_amp.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler_amp.step(opt); scaler_amp.update(); sched.step(); tot += loss.item()
            p_va = predict(model, dl_va)
            ap = average_precision_score(va.y, p_va)
            history.append(dict(epoch=ep + 1, train_loss=tot / len(dl_tr), val_pr_auc=ap))
            print(f"[{tag}] epoch {ep + 1}: loss={tot / len(dl_tr):.4f} val PR-AUC={ap:.3f} ({time.time() - t0:.0f}s)")
            if ap > best[0]:
                best = (ap, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, p_va)
        model.load_state_dict(best[1])
        return best[2], predict(model, dl_te), pd.DataFrame(history), model, tok

    tr_, va_, te_ = train.copy(), val.copy(), test.copy()
    if SMOKE_TEST:  # a few hundred sentences, just to exercise the code path
        tr_, va_, te_ = (pd.concat([f[f.y == 0].head(150), f[f.y == 1].head(150)]).reset_index(drop=True) for f in (train, val, test))

    # D: review only
    pv, pt, histD, modelD, tokD = train_transformer("D", tr_, va_, te_, max_len=64)
    va_["score_D"], te_["score_D"] = pv, pt

    # E: review + synopsis context
    for f in (tr_, va_, te_):
        f["syn_ctx"] = retrieve_context(f)
    print("example input pair:\n  ", te_.text.iloc[0], "\n  ", te_.syn_ctx.iloc[0][:300], "...")
    pv, pt, histE, modelE, tokE = train_transformer("E", tr_, va_, te_, pair_cols="syn_ctx", max_len=256)
    va_["score_E"], te_["score_E"] = pv, pt

    if not SMOKE_TEST:
        val["score_D"], test["score_D"] = va_.score_D.values, te_.score_D.values
        val["score_E"], test["score_E"] = va_.score_E.values, te_.score_E.values
        test["syn_ctx"] = te_.syn_ctx.values
        results.append(evaluate("D  DistilBERT (review only)", test, "score_D", best_f1_threshold(val.y, val.score_D))); SCORE_COLS["D"] = "score_D"
        results.append(evaluate("E  DistilBERT (review + synopsis)", test, "score_E", best_f1_threshold(val.y, val.score_E))); SCORE_COLS["E"] = "score_E"
        pd.concat([histD.assign(model="D"), histE.assign(model="E")]).to_csv(RESULTS / "distilbert_training_history.csv", index=False)
    else:
        print("smoke test finished: D val PR-AUC %.3f, E val PR-AUC %.3f (random tiny model, numbers meaningless)"
              % (average_precision_score(va_.y, va_.score_D), average_precision_score(va_.y, va_.score_E)))
else:
    print("No GPU: DistilBERT models skipped. Open in Colab with a T4 GPU to run them.")

## 7. Comparison of all models (test set, 21 unseen movies)

In [ ]:
res = pd.DataFrame(results).set_index("model")
res.round(3).to_csv(RESULTS / "metrics_test.csv")
res.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for key, col in SCORE_COLS.items():
    p, r, _ = precision_recall_curve(test.y, test[col])
    axes[0].plot(r, p, label=f"{key} (AP={average_precision_score(test.y, test[col]):.3f})")
axes[0].axhline(test.y.mean(), ls=":", c="grey", label=f"random ({test.y.mean():.3f})")
axes[0].set(xlabel="recall (spoilers caught)", ylabel="precision (masked sentences that are spoilers)",
            title="Precision-recall on test", xlim=(0, 1), ylim=(0, 1)); axes[0].legend()
# masking trade-off: share of words hidden vs share of spoiler sentences caught
for key, col in SCORE_COLS.items():
    ts = np.quantile(test[col], np.linspace(0, 1, 101))
    hidden = [(test.n_words * (test[col] >= t)).sum() / test.n_words.sum() for t in ts]
    caught = [((test[col] >= t) & (test.y == 1)).sum() / test.y.sum() for t in ts]
    axes[1].plot(hidden, caught, label=key)
axes[1].set(xlabel="share of review text hidden", ylabel="share of spoiler sentences hidden",
            title="Masking trade-off", xlim=(0, 0.5), ylim=(0, 1)); axes[1].legend()
plt.tight_layout(); plt.savefig(RESULTS / "pr_curves.png", dpi=130); plt.show()

## 8. Error analysis
Everything below uses the **best model by test PR-AUC**, and compares it with its review-only counterpart
to see *where* the synopsis helps or hurts.

In [ ]:
best_key = max(SCORE_COLS, key=lambda k: average_precision_score(test.y, test[SCORE_COLS[k]]))
pair = {"C": "A", "E": "D", "B": "A", "B0": "A"}.get(best_key, "A")
thr_of = {r["model"].split()[0]: r["threshold"] for r in results}
best_col, base_col = SCORE_COLS[best_key], SCORE_COLS[pair]
test["pred_best"] = test[best_col] >= thr_of[best_key]
test["pred_base"] = test[base_col] >= thr_of[pair]
print(f"best model: {best_key}, compared with review-only model {pair}")

### 8.1 Where do errors concentrate? (per movie, per sentence length, per review flag)

In [ ]:
def ap_safe(g, col):
    return average_precision_score(g.y, g[col]) if 0 < g.y.sum() < len(g) else np.nan

per_movie = test.groupby("movie_id").apply(lambda g: pd.Series({
    "sentences": len(g), "spoiler_rate": g.y.mean(),
    f"AP_{pair}": ap_safe(g, base_col), f"AP_{best_key}": ap_safe(g, best_col),
    "synopsis_sents": len(syn_sents[g.name])}), include_groups=False)
per_movie["gain"] = per_movie[f"AP_{best_key}"] - per_movie[f"AP_{pair}"]
title_map = {m: (synopses.get(m, {}).get("plot_summary") or "")[:60] for m in per_movie.index}
per_movie["plot_summary_start"] = per_movie.index.map(title_map)
per_movie.sort_values(f"AP_{best_key}").round(3).to_csv(RESULTS / "per_movie_ap.csv")
per_movie.sort_values(f"AP_{best_key}").round(3)

In [ ]:
test["len_bin"] = pd.cut(test.n_words, [0, 8, 15, 25, 40, 1000], labels=["<=8", "9-15", "16-25", "26-40", ">40"])
by_len = test.groupby("len_bin", observed=True).apply(lambda g: pd.Series({
    "n": len(g), "spoiler_rate": g.y.mean(), f"AP_{pair}": ap_safe(g, base_col), f"AP_{best_key}": ap_safe(g, best_col),
    f"recall_{best_key}": g[g.y == 1].pred_best.mean()}), include_groups=False)
by_flag = test.groupby("is_spoiler_review").apply(lambda g: pd.Series({
    "n": len(g), "spoiler_rate": g.y.mean(), f"AP_{best_key}": ap_safe(g, best_col),
    f"false_alarm_rate_{best_key}": g[g.y == 0].pred_best.mean()}), include_groups=False)
display(by_len.round(3))
by_flag.round(3)

### 8.2 Concrete failures
* **Missed spoilers** (false negatives with the lowest score): what kind of spoiler does the model not see?
* **False alarms** (false positives with the highest score): what looks like a spoiler but is not?
* **Synopsis helps / hurts**: sentences where the review-only and the synopsis model disagree.

The best-matching synopsis sentence and its plot position are shown, so you can judge whether matching worked.

In [ ]:
show_cols = ["movie_id", "text", best_col, base_col, "best_syn_pos", "best_syn_sentence"]
fn = test[(test.y == 1) & ~test.pred_best].sort_values(best_col).head(12)
fp = test[(test.y == 0) & test.pred_best].sort_values(best_col, ascending=False).head(12)
helps = test[(test.y == 1) & test.pred_best & ~test.pred_base].sort_values(best_col, ascending=False).head(8)
hurts = test[(test.y == 0) & test.pred_best & ~test.pred_base].sort_values(best_col, ascending=False).head(8)
for name, frame in [("missed_spoilers", fn), ("false_alarms", fp), ("synopsis_helps", helps), ("synopsis_hurts", hurts)]:
    frame[show_cols].round(3).to_csv(RESULTS / f"errors_{name}.csv", index=False)
print("MISSED SPOILERS (false negatives)")
fn[show_cols].round(3)

In [ ]:
print("FALSE ALARMS (false positives)")
fp[show_cols].round(3)

In [ ]:
print(f"SYNOPSIS HELPS: real spoilers caught by {best_key} but missed by {pair}")
display(helps[show_cols].round(3))
print(f"SYNOPSIS HURTS: false alarms introduced by {best_key}")
hurts[show_cols].round(3)

### 8.3 Label noise check
Train labels come from an LLM. How do models score the sentences the LLM itself marked `unsure`? (Validation only, since
test has no `unsure` flags.) If the model is torn exactly where the annotator was, part of the remaining error is
ambiguity in the task rather than a model weakness.

In [ ]:
print(val.groupby(["unsure", "y"]).agg(n=("y", "size"), **{f"mean_{k}": (c, "mean") for k, c in SCORE_COLS.items()}).round(3))

### 8.4 Stress tests: hand-written edge cases
Sentences written by us to probe specific behaviours, scored by every trained model. `expected` follows the
annotation guidelines (1 = spoiler). Movies are from the **test** split, so the models never saw their reviews.

| Probe | What it tests |
|---|---|
| explicit vs. paraphrased spoiler | does the model need names / keywords, or does it understand the event? |
| opinion about the ending, "no spoilers" disclaimers | keyword traps: "ending", "twist", "spoil" without content |
| premise vs. late event with the same character | does position in the plot matter? |
| wrong synopsis | a real spoiler scored against another movie's plot: how much does model C/E rely on the synopsis? |
| very short / context-dependent sentences | "He dies." has no names at all |

In [ ]:
STRESS = [
    # (movie_id, sentence, expected, probe)
    ("tt0468569", "Harvey Dent becomes Two-Face and dies at the end.", 1, "explicit late spoiler"),
    ("tt0468569", "The prosecutor who was the city's great hope turns into a killer himself.", 1, "paraphrased spoiler, no names"),
    ("tt0468569", "Rachel does not survive.", 1, "negation spoiler"),
    ("tt0468569", "Batman, Gordon and Harvey Dent team up to clean up Gotham's organised crime.", 0, "premise with the same names"),
    ("tt0468569", "Heath Ledger is unforgettable as the Joker.", 0, "actor / role"),
    ("tt0468569", "I won't spoil anything, but the ending is great.", 0, "'no spoilers' disclaimer"),
    ("tt0468569", "The ending was weak and far too long.", 0, "opinion about the ending"),
    ("tt0468569", "There is a huge twist in the last act that changes everything.", 1, "existence of a twist (guideline)"),
    ("tt0167260", "Frodo and Sam continue their journey to Mordor to destroy the Ring.", 0, "premise"),
    ("tt0167260", "Gollum bites off Frodo's finger and falls into the fire with the Ring.", 1, "explicit late spoiler"),
    ("tt0167260", "In the end Sam goes home and marries Rosie.", 1, "explicit ending"),
    ("tt0167260", "Lots of orcs die in the big battle.", 0, "generic deaths in a war film (guideline)"),
    ("tt0137523", "The narrator and Tyler are actually the same person.", 1, "famous twist"),
    ("tt0137523", "Brad Pitt plays Tyler Durden.", 0, "actor / role"),
    ("tt0137523", "The rules of fight club are the best scene of the film.", 0, "opinion with plot words"),
    ("tt0068646", "Michael becomes the new Don after his father dies.", 1, "character transformation"),
    ("tt0068646", "Marlon Brando is brilliant as Vito Corleone.", 0, "actor / role"),
    ("tt1355644", "Two passengers wake up 90 years too early on a spaceship.", 0, "premise (guideline example)"),
    ("tt1355644", "Jim wakes Aurora up on purpose because he is lonely.", 1, "mid-film reveal"),
    ("tt0468569", "He dies.", 1, "very short, no names"),
    ("tt0167260", "Harvey Dent becomes Two-Face and dies at the end.", 1, "WRONG synopsis (LOTR) for a Dark Knight spoiler"),
]
stress = pd.DataFrame(STRESS, columns=["movie_id", "text", "expected", "probe"])
stress["n_words"] = stress.text.str.split().str.len()


def score_frame(frame):
    """Score arbitrary sentences with every model trained in this run."""
    out = {}
    F, idx = synopsis_features(frame)
    XA, FS = tfidf(frame), scaler.transform(F)
    out["A"] = lrA.predict_proba(XA)[:, 1]
    out["B"] = lrB.predict_proba(FS)[:, 1]
    out["C"] = lrC.predict_proba(sparse.hstack([XA, sparse.csr_matrix(FS * wC)]).tocsr())[:, 1]
    if "D" in SCORE_COLS and "modelD" in globals():
        import torch
        def tscore(model, tok, texts, pairs=None, max_len=64):
            model.eval()
            with torch.no_grad():
                b = tok(texts, pairs, truncation="longest_first" if pairs else True, max_length=max_len,
                        padding=True, return_tensors="pt").to(DEVICE)
                return torch.softmax(model(**b).logits.float(), -1)[:, 1].cpu().numpy()
        out["D"] = tscore(modelD, tokD, frame.text.tolist())
        out["E"] = tscore(modelE, tokE, frame.text.tolist(), retrieve_context(frame), max_len=256)
    return out, [syn_sents[m][i] if syn_sents[m] else ("", 0) for m, i in zip(frame.movie_id, idx)]


scores, matched = score_frame(stress)
for k, v in scores.items():
    stress[f"score_{k}"] = v.round(3)
    stress[f"ok_{k}"] = (v >= thr_of[k]) == (stress.expected == 1)
stress["matched_plot_pos"] = [round(p, 2) for _, p in matched]
stress["matched_plot_sentence"] = [s[:110] for s, _ in matched]
stress.to_csv(RESULTS / "stress_tests.csv", index=False)
print("share of stress cases handled correctly:",
      {k: round(stress[f"ok_{k}"].mean(), 2) for k in scores})
stress[["probe", "text", "expected"] + [f"score_{k}" for k in scores] + [f"ok_{k}" for k in scores] + ["matched_plot_pos"]]

## 9. Masking demo
The end product: mask the sentences above the threshold and show *why*, i.e. which part of the plot the sentence
gives away. Uses the classical model C (runs anywhere, no GPU).

In [ ]:
def mask_review(review_text, movie_id, model_key="C"):
    sents = split_sentences(review_text)
    frame = pd.DataFrame({"text": sents, "movie_id": movie_id})
    frame["n_words"] = frame.text.str.split().str.len()
    F, idx = synopsis_features(frame)
    XA = tfidf(frame)
    FS = scaler.transform(F)
    score = lrC.predict_proba(sparse.hstack([XA, sparse.csr_matrix(FS * wC)]).tocsr())[:, 1] if model_key == "C" \
        else lrA.predict_proba(XA)[:, 1]
    thr = thr_of[model_key]
    out = []
    for s, sc, i in zip(sents, score, idx):
        if sc >= thr:
            syn, pos = syn_sents[movie_id][i] if syn_sents[movie_id] else ("", 0)
            out.append(f"█ [SPOILER {sc:.2f}] (matches plot at {pos:.0%}: \"{syn[:90]}...\")")
        else:
            out.append(s)
    return "\n".join(out)


demo = test[test.review_id == test[(test.y == 1)].review_id.iloc[3]]
print("ORIGINAL (labels in brackets):\n" + "\n".join(f"[{'S' if y else ' '}] {t}" for t, y in zip(demo.text, demo.y)))
print("\nMASKED:\n" + mask_review(" ".join(demo.text), demo.movie_id.iloc[0]))

## 10. Save predictions
Test-set predictions of every model, for the report and for anyone who wants to inspect the errors without running anything.

In [ ]:
keep = ["review_id", "movie_id", "s_idx", "text", "y", "is_spoiler_review", "best_syn_pos", "best_syn_sentence"] + list(SCORE_COLS.values())
test[keep].round(4).to_csv(RESULTS / "predictions_test.csv", index=False)
json.dump({"thresholds": thr_of, "best_model": best_key, "synopsis_weight_C": wC,
           "val_movies": sorted(val.movie_id.unique())}, open(RESULTS / "run_config.json", "w"), indent=2)
print("saved to", RESULTS)